<a href="https://colab.research.google.com/github/santi-boe/Document-Compiler/blob/main/Master_Note_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2
import os
import requests
import PyPDF2
from PyPDF2 import PdfMerger
from google.colab import auth, drive

In [ ]:
from google.colab import auth, drive
import os

auth.authenticate_user()

drive.mount('/content/drive')

drive_path = '/content/drive'
output_drive_folder = os.path.join(drive_path, 'My Drive', 'Colab Notebooks')
output_filename = 'Chapter_Notes_Master.pdf'
final_output_path = os.path.join(output_drive_folder, output_filename)

os.makedirs(output_drive_folder, exist_ok=True)
print(f"Output PDF will be saved to: {final_output_path}")

In [ ]:
# List of URLs
# Replace these with your actual URLs
google_slides_urls = [
    # paste all links here inside a quotation
    # all links must contain /export/pdf at the end
]

temp_pdf_dir = '/content/temp_slides_pdfs'
os.makedirs(temp_pdf_dir, exist_ok=True)
print(f"Temporary PDF files will be stored in: {temp_pdf_dir}")

Temporary PDF files will be stored in: /content/temp_slides_pdfs


In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

def download_slides_as_pdf_v2(file_id, output_path):
    service = build('drive', 'v3')
    try:
        request = service.files().export_media(fileId=file_id, mimeType='application/pdf')
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while done is False:
            status, done = downloader.next_chunk()

        with open(output_path, 'wb') as f:
            f.write(fh.getvalue())
        print(f"Successfully downloaded ID: {file_id}")
        return True
    except Exception as e:
        print(f"Error downloading {file_id}: {e}")
        return False

In [ ]:
pdf_files_to_merge = []

print("Starting downloads...")

for i, url in enumerate(google_slides_urls):
    try:
        file_id = url.split('/d/')[1].split('/')[0]
        output_pdf_path = os.path.join(temp_pdf_dir, f'slide_{i+1}.pdf')

        if download_slides_as_pdf_v2(file_id, output_pdf_path):
            pdf_files_to_merge.append(output_pdf_path)
    except Exception as e:
        print(f"Skipping link {i+1} due to error: {e}")

if not pdf_files_to_merge:
    print("\nNo PDFs were downloaded. Check your permissions.")
else:
    print(f"\nMerging {len(pdf_files_to_merge)} PDFs...")
    merger = PdfMerger()
    for pdf_file in pdf_files_to_merge:
        merger.append(pdf_file)

    merger.write(final_output_path)
    merger.close()
    print(f"\nDONE! Your giant PDF is at: {final_output_path}")